# Read Data from a Dataverse Table

A simple notebook that retrieves data from a Dataverse table using the **Web API (REST)** and displays it as a table.

**Why REST?** To read records you send a `GET` request to the entity set resource (e.g. `/api/data/v9.2/accounts`). You can shape the result with [OData query options](https://learn.microsoft.com/power-apps/developer/data-platform/webapi/query-data-web-api) such as `$select`, `$filter`, `$top` and `$orderby`.

In [ ]:
import requests
import pandas as pd
import notebookutils

In [ ]:
# --- Configuration ---
dataverse_env_url = "https://org0f651234be2.crm4.dynamics.com"

tenant_id  = "9e929790-272d-4977-a2ab-301443c11ece"
client_id  = "b5c04c9c-0588-418f-8f60-2d83d38cb635"

azure_key_vault_name = "ab-fabric-automation-akv"
secret_name          = "fabric-monit-secret"

# Entity set name of the target Dataverse table (plural). Change to your table.
entity_set_name = "accounts"
api_version     = "v9.2"

In [ ]:
# --- Get the service principal secret from Key Vault ---
azure_key_vault_url = f"https://{azure_key_vault_name}.vault.azure.net"
client_secret = notebookutils.credentials.getSecret(azure_key_vault_url, secret_name)

In [ ]:
# --- Acquire a Dataverse access token (client credentials / SPN) ---
def get_dataverse_token():
    url = f"https://login.microsoftonline.com/{tenant_id}/oauth2/v2.0/token"
    payload = {
        "grant_type":    "client_credentials",
        "client_id":     client_id,
        "client_secret": client_secret,
        "scope":         dataverse_env_url.rstrip("/") + "/.default",
    }
    response = requests.post(url, data=payload)
    response.raise_for_status()
    return response.json()["access_token"]

dataverse_token = get_dataverse_token()
print("Token acquired.")

In [ ]:
# --- Read records via GET (with optional OData query options) ---
def get_records(select=None, filter=None, top=None, orderby=None):
    url = f"{dataverse_env_url.rstrip('/')}/api/data/{api_version}/{entity_set_name}"
    params = {}
    if select:
        params["$select"] = select
    if filter:
        params["$filter"] = filter
    if top:
        params["$top"] = top
    if orderby:
        params["$orderby"] = orderby

    headers = {
        "Authorization":    f"Bearer {dataverse_token}",
        "OData-MaxVersion": "4.0",
        "OData-Version":    "4.0",
        "Accept":           "application/json",
    }
    response = requests.get(url, headers=headers, params=params)
    response.raise_for_status()
    return response.json().get("value", [])

In [ ]:
# --- Fetch and display the data ---
records = get_records(
    select="name,address1_city",
    top=10,
    orderby="name asc",
)

df = pd.DataFrame(records)
print(f"Retrieved {len(df)} record(s).")
display(df)